# Análise Exploratória: Sistema de Retenção de Talentos (People Analytics)

## 1. Contexto e Objetivo

A rotatividade de funcionários gera custos diretos (recrutamento, treinamento, onboarding) e indiretos (perda de produtividade, impacto no clima e conhecimento tácito). Desligamentos não planejados podem afetar resultados operacionais e financeiros, especialmente em áreas estratégicas.

O projeto visa criar um modelo preditivo para estimar a probabilidade de desligamento voluntário de funcionários ativos, usando informações demográficas, funcionais e de engajamento, para apoiar a priorização de ações de retenção.

**Pergunta foco:** Quais funcionários apresentam maior risco relativo de desligamento, permitindo direcionar ações preventivas de forma estratégica?

*Observação:* O modelo serve como ferramenta de apoio à decisão, sem caráter punitivo ou automático.

## 2. Carregamento dos Dados

In [5]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.float_format = '{:.2f}'.format

In [14]:
dados = pd.read_csv("../data/raw/IBM_HR_Employee_Attrition.csv")

# Visualizar primeiras linhas
dados.head(1)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5


In [15]:
# Traduzir os nomes das variáveis, facilitando a interpretação

# Dicionário com tradução das colunas
dicionario_traducao = {
    "Age": "idade",
    "Attrition": "desligamento",
    "BusinessTravel": "viagem_a_trabalho",
    "DailyRate": "taxa_diaria",
    "Department": "departamento",
    "DistanceFromHome": "distancia_de_casa",
    "Education": "educacao",
    "EducationField": "area_de_educacao",
    "EmployeeCount": "contagem_de_empregados",
    "EmployeeNumber": "numero_do_empregado",
    "EnvironmentSatisfaction": "satisfacao_ambiente",
    "Gender": "genero",
    "HourlyRate": "taxa_horaria",
    "JobInvolvement": "envolvimento_no_trabalho",
    "JobLevel": "nivel_do_trabalho",
    "JobRole": "funcao",
    "JobSatisfaction": "satisfacao_no_trabalho",
    "MaritalStatus": "estado_civil",
    "MonthlyIncome": "renda_mensal",
    "MonthlyRate": "taxa_mensal",
    "NumCompaniesWorked": "num_empresas_trabalhadas",
    "Over18": "mais_de_18_anos",
    "OverTime": "horas_extras",
    "PercentSalaryHike": "percentual_aumento_salarial",
    "PerformanceRating": "nota_de_desempenho",
    "RelationshipSatisfaction": "satisfacao_relacionamento",
    "StandardHours": "horas_padrao",
    "StockOptionLevel": "nivel_opcoes_acoes",
    "TotalWorkingYears": "anos_totais_de_trabalho",
    "TrainingTimesLastYear": "treinamentos_ultimo_ano",
    "WorkLifeBalance": "equilibrio_vida_trabalho",
    "YearsAtCompany": "anos_na_empresa",
    "YearsInCurrentRole": "anos_na_funcao_atual",
    "YearsSinceLastPromotion": "anos_desde_ultima_promocao",
    "YearsWithCurrManager": "anos_com_gerente_atual"
}

dados = dados.rename(columns=dicionario_traducao)
dados.columns

Index(['idade', 'desligamento', 'viagem_a_trabalho', 'taxa_diaria',
       'departamento', 'distancia_de_casa', 'educacao', 'area_de_educacao',
       'contagem_de_empregados', 'numero_do_empregado', 'satisfacao_ambiente',
       'genero', 'taxa_horaria', 'envolvimento_no_trabalho',
       'nivel_do_trabalho', 'funcao', 'satisfacao_no_trabalho', 'estado_civil',
       'renda_mensal', 'taxa_mensal', 'num_empresas_trabalhadas',
       'mais_de_18_anos', 'horas_extras', 'percentual_aumento_salarial',
       'nota_de_desempenho', 'satisfacao_relacionamento', 'horas_padrao',
       'nivel_opcoes_acoes', 'anos_totais_de_trabalho',
       'treinamentos_ultimo_ano', 'equilibrio_vida_trabalho',
       'anos_na_empresa', 'anos_na_funcao_atual', 'anos_desde_ultima_promocao',
       'anos_com_gerente_atual'],
      dtype='object')

**⚠️ Importante:**
*Assumimos o nome da target como "desligamento", no entanto, reforça-se que a variável indica apenas se o colaborador não está mais na empresa, sem distinção entre desligamento voluntário e involuntário.
Dessa forma, o modelo desenvolvido neste projeto não prevê exclusivamente pedidos de demissão, mas sim o risco de perda de colaboradores, o que deve ser considerado na interpretação dos resultados e no uso gerencial do sistema.*

### 2.1 Dicionário de Tradução dos Valores Categóricos

| Variável                  | Código | Descrição Original | Descrição em Português    |
|--------------------------|--------|--------------------|--------------------------|
| **educacao**              | 1      | Below College      | Ensino Fundamental       |
|                          | 2      | College            | Ensino Médio             |
|                          | 3      | Bachelor           | Graduação                |
|                          | 4      | Master             | Mestrado                 |
|                          | 5      | Doctor             | Doutorado                |
| **satisfacao_ambiente**   | 1      | Low                | Baixa                    |
|                          | 2      | Medium             | Média                    |
|                          | 3      | High               | Alta                     |
|                          | 4      | Very High          | Muito Alta               |
| **envolvimento_no_trabalho** | 1      | Low                | Baixo                    |
|                          | 2      | Medium             | Médio                    |
|                          | 3      | High               | Alto                     |
|                          | 4      | Very High          | Muito Alto               |
| **satisfacao_no_trabalho**| 1      | Low                | Baixa                    |
|                          | 2      | Medium             | Média                    |
|                          | 3      | High               | Alta                     |
|                          | 4      | Very High          | Muito Alta               |
| **nota_de_desempenho**    | 1      | Low                | Baixa                    |
|                          | 2      | Good               | Boa                      |
|                          | 3      | Excellent          | Excelente                |
|                          | 4      | Outstanding        | Excepcional              |
| **satisfacao_relacionamento** | 1      | Low                | Baixa                    |
|                          | 2      | Medium             | Média                    |
|                          | 3      | High               | Alta                     |
|                          | 4      | Very High          | Muito Alta               |
| **equilibrio_vida_trabalho** | 1      | Bad                | Ruim                     |
|                          | 2      | Good               | Boa                      |
|                          | 3      | Better             | Melhor                   |
|                          | 4      | Best               | Ótima                    |

## 3. Visão Geral e Qualidade dos Dados

In [16]:
print(f"Número de linhas: {dados.shape[0]}")
print(f"Número de colunas: {dados.shape[1]}")

dados.info()

Número de linhas: 1470
Número de colunas: 35
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   idade                        1470 non-null   int64 
 1   desligamento                 1470 non-null   object
 2   viagem_a_trabalho            1470 non-null   object
 3   taxa_diaria                  1470 non-null   int64 
 4   departamento                 1470 non-null   object
 5   distancia_de_casa            1470 non-null   int64 
 6   educacao                     1470 non-null   int64 
 7   area_de_educacao             1470 non-null   object
 8   contagem_de_empregados       1470 non-null   int64 
 9   numero_do_empregado          1470 non-null   int64 
 10  satisfacao_ambiente          1470 non-null   int64 
 11  genero                       1470 non-null   object
 12  taxa_horaria                 1470 non-null   

In [17]:
# Uma vez que não há valores nulos, verificaremos possíveis duplicidades existentes

duplicados = dados.duplicated().sum()
print(f"Linhas duplicadas: {duplicados}")

Linhas duplicadas: 0


**Dataset público e já pré-processado, o que reduz esforço de limpeza, mas limita a representatividade de dados reais.**

In [19]:
# Remoção de variáveis sem valor analítico

colunas_remover = [
    "contagem_de_empregados",
    "horas_padrao",
    "mais_de_18_anos",
    "numero_do_empregado"
]

dados = dados.drop(columns=colunas_remover)

# Conferência
dados.shape

(1470, 31)

**ℹ️ Justificativa da exclusão de variáveis:**
*As variáveis removidas são identificadores ou apresentam valor constante ao longo da base, não contribuindo para a explicação do risco de desligamento nem para a tomada de decisão gerencial. Sua exclusão reduz ruído e melhora a interpretabilidade das análises e modelos subsequentes.*

In [23]:
distribuicao_desligamento = (dados["desligamento"]
                             .value_counts(normalize=True)
                             .rename("percentual")
                             .reset_index()
                            )

distribuicao_desligamento["percentual"] = (distribuicao_desligamento["percentual"] * 100).round(2)
distribuicao_desligamento

,desligamento,percentual
0,No,83.88
1,Yes,16.12


### 3.1 Hipóteses de negócio: 
#### 3.1.1 - **Horas extras aumentam risco de desligamento?**

In [25]:
analise_horas_extras_qtd = (dados.groupby("horas_extras")["desligamento"]
                             .value_counts()
                             .rename("qtd")
                             .reset_index()
                       )

analise_horas_extras_qtd

,horas_extras,desligamento,qtd
0,No,No,944
1,No,Yes,110
2,Yes,No,289
3,Yes,Yes,127


In [24]:
analise_horas_extras = (dados.groupby("horas_extras")["desligamento"]
                             .value_counts(normalize=True)
                             .rename("percentual")
                             .reset_index()
                       )

analise_horas_extras["percentual"] = (analise_horas_extras["percentual"] * 100).round(2)

analise_horas_extras

,horas_extras,desligamento,percentual
0,No,No,89.56
1,No,Yes,10.44
2,Yes,No,69.47
3,Yes,Yes,30.53


⚠️ **Observações:**
- 10,5% de quem NÃO teve hora extra, tem registro de desligamento
- 30,5% de quem TEM hora extra, tem registro de desligamento
- **Aproximadamente 3x mais**

---
- Quase 70% das pessoas que fizeram hora extra não tiveram desligamento

*Isso pode indicar que horas extras não são causa comprovada, mas são um forte indicador associado ao desligamento*

---
- Em números absolutos: 
    - 1.233 funcionários sem desligamento, desses 944 sem fazer horas extras (76,5%)
    - 237 funcionários com desligamento, desses 127 fazendo horas extras (53,5%)
    - **Mais da metade dos desligamentos, tem ocorrência de horas extras realizadas**
---
**Colaboradores que realizam horas extras apresentam um risco de desligamento aproximadamente três vezes maior. Embora a maioria permaneça na empresa, esse grupo concentra uma parcela desproporcional dos desligamentos, tornando-se um público prioritário para ações preventivas de gestão de carga de trabalho, acompanhamento e engajamento.**

#### 3.1.2 - **Hipótese de negócio 2: Satisfação no trabalho está associada ao desligamento?**

In [28]:
analise_satisfacao_qtd = (dados.groupby("satisfacao_no_trabalho")["desligamento"]
                               .value_counts()
                               .rename("qtd")
                               .reset_index()
                         )

analise_satisfacao_qtd.sort_values("satisfacao_no_trabalho")

,satisfacao_no_trabalho,desligamento,qtd
0,1,No,223
1,1,Yes,66
2,2,No,234
3,2,Yes,46
4,3,No,369
5,3,Yes,73
6,4,No,407
7,4,Yes,52


In [27]:
analise_satisfacao = (dados.groupby("satisfacao_no_trabalho")["desligamento"]
                           .value_counts(normalize=True)
                           .rename("percentual")
                           .reset_index()
                     )

analise_satisfacao["percentual"] = (analise_satisfacao["percentual"] * 100).round(2)

analise_satisfacao.sort_values("satisfacao_no_trabalho")

,satisfacao_no_trabalho,desligamento,percentual
0,1,No,77.16
1,1,Yes,22.84
2,2,No,83.57
3,2,Yes,16.43
4,3,No,83.48
5,3,Yes,16.52
6,4,No,88.67
7,4,Yes,11.33


⚠️ **Observações:**
*Embora colaboradores com menor satisfação apresentem maior risco relativo de desligamento, observa-se que níveis intermediários de satisfação concentram maior número absoluto de desligamentos. 
Isso sugere a necessidade de estratégias combinadas: **ações focalizadas em grupos de alto risco proporcional** e **iniciativas sistêmicas voltadas a grupos numericamente maiores.***

#### 3.1.3 - **Hipótese de negócio 3: Renda mensal e nível do trabalho influenciam o desligamento?**

In [29]:
# Optei por criar faixas de renda, pois fica melhor para visualizarmos (como se fossem sub-grupos)
# O método de cortes em quartis, proporciona uma divisão balanceada, com uma comparação mais robusta

dados["faixa_renda"] = pd.qcut(dados["renda_mensal"], q=4, 
                               labels=["baixa", "media_baixa", "media_alta", "alta"]
                              )
dados["faixa_renda"].value_counts()

faixa_renda
baixa          369
alta           368
media_alta     367
media_baixa    366
Name: count, dtype: int64

In [37]:
analise_renda_qtd = (dados.groupby("faixa_renda", observed=False)["desligamento"]
                     .value_counts()
                     .rename("qtd")
                     .reset_index()
                    )

analise_renda_qtd

,faixa_renda,desligamento,qtd
0,baixa,No,261
1,baixa,Yes,108
2,media_baixa,No,314
3,media_baixa,Yes,52
4,media_alta,No,328
5,media_alta,Yes,39
6,alta,No,330
7,alta,Yes,38


In [38]:
analise_renda = (dados.groupby("faixa_renda", observed=False)["desligamento"]
                      .value_counts(normalize=True)
                      .rename("percentual")
                      .reset_index()
                )

analise_renda["percentual"] = (analise_renda["percentual"] * 100).round(2)

analise_renda

,faixa_renda,desligamento,percentual
0,baixa,No,70.73
1,baixa,Yes,29.27
2,media_baixa,No,85.79
3,media_baixa,Yes,14.21
4,media_alta,No,89.37
5,media_alta,Yes,10.63
6,alta,No,89.67
7,alta,Yes,10.33


⚠️ **Observações:**
*Colaboradores nas faixas de menor renda apresentam risco de desligamento significativamente superior, sugerindo que remuneração e/ou perspectivas de crescimento são fatores relevantes na decisão de saída.*

*Isso pode sugerir inclusive, ações estratégicas:*
- Revisão de política salarial para faixas mais baixas
- Priorização de programas de progressão de carreira
- Monitoramento ativo de colaboradores em faixas iniciais

*Em resumo, observa-se um gradiente claro entre renda mensal e desligamento, com colaboradores na faixa de menor renda apresentando risco quase três vezes superior ao observado nas faixas mais altas. Esse padrão sugere que remuneração e perspectivas de crescimento podem desempenhar papel relevante na retenção, reforçando a importância de ações direcionadas a esse grupo.*

#### 3.1.4 - **Hipótese de negócio 4: Nível do trabalho (senioridade) influencia o desligamento?**

In [41]:
dados['nivel_do_trabalho'].value_counts()

nivel_do_trabalho
1    543
2    534
3    218
4    106
5     69
Name: count, dtype: int64

**Vamos assumir esses níveis, para nossa análise**
- 1 = Nível Junior
- 2 = Nível Pleno I
- 3 = Nível Pleno II
- 4 = Nível Senior
- 5 = Nível Executivo/Gestor

In [39]:
analise_nivel_qtd = (dados.groupby("nivel_do_trabalho")["desligamento"]
                      .value_counts()
                      .rename("qtd")
                      .reset_index()
                )

analise_nivel_qtd.sort_values("nivel_do_trabalho")

,nivel_do_trabalho,desligamento,qtd
0,1,No,400
1,1,Yes,143
2,2,No,482
3,2,Yes,52
4,3,No,186
5,3,Yes,32
6,4,No,101
7,4,Yes,5
8,5,No,64
9,5,Yes,5


In [42]:
analise_nivel = (dados.groupby("nivel_do_trabalho")["desligamento"]
                      .value_counts(normalize=True)
                      .rename("percentual")
                      .reset_index()
                )

analise_nivel["percentual"] = (analise_nivel["percentual"] * 100).round(2)

analise_nivel.sort_values("nivel_do_trabalho")

,nivel_do_trabalho,desligamento,percentual
0,1,No,73.66
1,1,Yes,26.34
2,2,No,90.26
3,2,Yes,9.74
4,3,No,85.32
5,3,Yes,14.68
6,4,No,95.28
7,4,Yes,4.72
8,5,No,92.75
9,5,Yes,7.25


⚠️ **Observações:** 

***Nomes que eu coloquei***
 - 1 --> Total de Colaboradores 543 - Junior **(143 desligamentos)**
 - 2 --> Total de Colaboradores 534 - Pleno I **(52 desligamentos)**
 - 3 --> Total de Colaboradores 218 - Pleno II **(32 desligamentos)**
 - 4 --> Total de Colaboradores 106 - Senior **(5 desligamentos)**
 - 5 --> Total de Colaboradores 69 - Gestor **(5 desligamentos)**

---
- *Níveis menores concentram em torno de 73% dos colaboradores, isso tende a representar maioria quando analisados os desligamentos em números absolutos*
 - *O nível 1 apresenta sim a maior taxa de desligamento -- 26,34% -- apesar de não se ter causa, mas podemos considerar fatores, como por exemplo: carreira em ascenção, busca por melhor remuneração ou por crescimento acelerado*
- *O nível 3 (14,68%) apresentou taxa superior ao nível 2 (9,74%), em comparação percentual*
-  *Os demais níveis apresentam comportamento reduzido de desligamento (nível 4: 4,72% e nível 5: 7,25%)*
---
*Uma hipótese plausível é que colaboradores em níveis intermediários enfrentem maior fricção de progressão na carreira, o que pode elevar o risco de desligamento mesmo quando comparado ao nível imediatamente anterior.*

#### 3.1.5 - **Hipótese 5: Tempo desde última promoção influencia desligamento?**

In [48]:
dados["anos_desde_ultima_promocao"].describe()

count   1470.00
mean       2.19
std        3.22
min        0.00
25%        0.00
50%        1.00
75%        3.00
max       15.00
Name: anos_desde_ultima_promocao, dtype: float64

In [43]:
bins = [-1, 0, 1, 2, 4, 10, 50]
labels = ["0", "1", "2", "3-4", "5-10", "11+"]

dados["faixa_tempo_promocao"] = pd.cut(dados["anos_desde_ultima_promocao"],
                                       bins=bins,
                                       labels=labels
                                      )
dados["faixa_tempo_promocao"].value_counts()

faixa_tempo_promocao
0       581
1       357
5-10    194
2       159
3-4     113
11+      66
Name: count, dtype: int64

In [45]:
analise_promocao_qtd = (dados.groupby("faixa_tempo_promocao", observed=False)["desligamento"]
                             .value_counts()
                             .rename("qtd")
                             .reset_index()
                       )

analise_promocao_qtd

,faixa_tempo_promocao,desligamento,qtd
0,0,No,471
1,0,Yes,110
2,1,No,308
3,1,Yes,49
4,2,No,132
5,2,Yes,27
6,3-4,No,99
7,3-4,Yes,14
8,5-10,No,165
9,5-10,Yes,29


In [47]:
analise_promocao = (dados.groupby("faixa_tempo_promocao", observed=False)["desligamento"]
                         .value_counts(normalize=True)
                         .rename("percentual")
                         .reset_index()
                   )

analise_promocao["percentual"] = (analise_promocao["percentual"] * 100).round(2)

analise_promocao

,faixa_tempo_promocao,desligamento,percentual
0,0,No,81.07
1,0,Yes,18.93
2,1,No,86.27
3,1,Yes,13.73
4,2,No,83.02
5,2,Yes,16.98
6,3-4,No,87.61
7,3-4,Yes,12.39
8,5-10,No,85.05
9,5-10,Yes,14.95


⚠️ **Observações:**

*Temos uma quantidade considerável para a faixa de promoções recentemente efetivadas (grupo 0): 581 pessoas. Dessas pessoas cerca de 19% efetivaram desligamento --> 110 / 18,93% **representando a maior faixa com taxa de desligamento, seja em números absolutos, uma vez que é o maior grupo, ou por percentual proporcional.** 
Interessante perceber que a menor taxa de desligamento está no grupo 11+, com 12,12%. Pode ser um sinal de conformismo ou estagnação.*

*Colaboradores com promoção recente (0 anos desde última promoção) apresentam a maior taxa de desligamento. Isso sugere que **a promoção isoladamente pode não ser suficiente como ação de retenção e reforça a importância de acompanhamento pós-promoção (adaptação, carga de trabalho e expectativas).***

⚠️ **Limitação:** *o dataset é estático e não permite inferir causalidade nem sequência temporal entre promoção e desligamento.*

## 4. Aprofundar resultado das Hipóteses Testadas

### 4.1 Tabela para visualização unica de horas extras, renda e senioridade

In [62]:
'''
- Taxa de desligamento
- Tamanho do grupo
- Desligamentos
'''

def tabela_risco_por_grupo(dados, coluna_grupo, coluna_alvo="desligamento"):
    tabela = (
        dados
        .groupby(coluna_grupo, observed=False)[coluna_alvo]
        .value_counts()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Garantir colunas Yes/No
    if "Yes" not in tabela.columns:
        tabela["Yes"] = 0
    if "No" not in tabela.columns:
        tabela["No"] = 0

    tabela["total"] = tabela["Yes"] + tabela["No"]
    tabela["taxa_desligamento"] = ((tabela["Yes"] / tabela["total"])*100).round(4)

    tabela = tabela.rename(columns={
        coluna_grupo: "grupo",
        "Yes": "qtd_desligamentos",
        "No": "qtd_nao_desligamentos"
    })

    tabela = tabela.sort_values("taxa_desligamento", ascending=False)

    return tabela[["grupo", "total", "qtd_desligamentos", "taxa_desligamento"]]


# Tabelas principais
tabela_horas_extras = tabela_risco_por_grupo(dados, "horas_extras")
tabela_faixa_renda = tabela_risco_por_grupo(dados, "faixa_renda")
tabela_nivel_trabalho = tabela_risco_por_grupo(dados, "nivel_do_trabalho")
tabela_satisfacao_trabalho = tabela_risco_por_grupo(dados, "satisfacao_no_trabalho")

print("***** TABELA DE HORAS EXTRAS *****")
display(tabela_horas_extras)

print("\n***** TABELA DE RENDA, POR FAIXAS *****")
display(tabela_faixa_renda)

print("\n***** TABELA DE NÍVEL DO TRABALHO - SENIORIDADE *****")
display(tabela_nivel_trabalho)

***** TABELA DE HORAS EXTRAS *****


desligamento,grupo,total,qtd_desligamentos,taxa_desligamento
1,Yes,416,127,30.53
0,No,1054,110,10.44



***** TABELA DE RENDA, POR FAIXAS *****


desligamento,grupo,total,qtd_desligamentos,taxa_desligamento
0,baixa,369,108,29.27
1,media_baixa,366,52,14.21
2,media_alta,367,39,10.63
3,alta,368,38,10.33



***** TABELA DE NÍVEL DO TRABALHO - SENIORIDADE *****


desligamento,grupo,total,qtd_desligamentos,taxa_desligamento
0,1,543,143,26.34
2,3,218,32,14.68
1,2,534,52,9.74
4,5,69,5,7.25
3,4,106,5,4.72


### 4.2 Combinando perfis de risco, com base nos três segmentos analisados nas hipóteses

In [64]:
colunas_segmento = ["horas_extras", "faixa_renda", "nivel_do_trabalho"]

# Criar um perfil concatenado para segmentação
dados["perfil_risco"] = (
    "horas_extras=" + dados["horas_extras"].astype(str) + " | " +
    "faixa_renda=" + dados["faixa_renda"].astype(str) + " | " +
    "nivel=" + dados["nivel_do_trabalho"].astype(str)
)

# Tabela de risco por perfil
tabela_perfis = tabela_risco_por_grupo(dados, "perfil_risco")

# Filtro para perfis com volume mínimo (evitar perfis raros e instáveis)
tabela_perfis_filtrada = tabela_perfis[tabela_perfis["total"] >= 30].reset_index(drop=True)

tabela_perfis_filtrada.head(10)

desligamento,grupo,total,qtd_desligamentos,taxa_desligamento
0,horas_extras=Yes | faixa_renda=baixa | nivel=1,102,62,60.78
1,horas_extras=Yes | faixa_renda=media_baixa | nivel=1,52,19,36.54
2,horas_extras=Yes | faixa_renda=alta | nivel=3,48,11,22.92
3,horas_extras=Yes | faixa_renda=media_alta | nivel=2,92,18,19.57
4,horas_extras=No | faixa_renda=media_alta | nivel=3,39,7,17.95
5,horas_extras=No | faixa_renda=baixa | nivel=1,254,45,17.72
6,horas_extras=Yes | faixa_renda=media_baixa | nivel=2,44,6,13.64
7,horas_extras=No | faixa_renda=media_baixa | nivel=1,131,16,12.21
8,horas_extras=No | faixa_renda=alta | nivel=3,116,12,10.34
9,horas_extras=Yes | faixa_renda=alta | nivel=4,33,3,9.09


## 5. Conclusões da EDA (Resumo Executivo)

A análise exploratória orientada a hipóteses de negócio identificou padrões consistentes associados ao desligamento, com potencial direto de apoiar decisões preventivas de retenção. Os principais achados foram:

- **Horas extras** apresentou associação forte com desligamento, com aumento relevante na taxa de saída entre colaboradores que realizam jornada extra.
- **Renda mensal (faixa baixa)** demonstrou aspcetos claros de risco, sugerindo maior vulnerabilidade de retenção em grupos com menor remuneração.
- **Nível do cargo (nível 1)** concentrou maior taxa e volume de desligamentos, indicando maior risco em estágios iniciais de carreira.
- **Satisfação no trabalho** apresentou relação com desligamento, porém com alguns desvios: grupos na faixa intermediária concentraram quantidades relevantes de desligamentos.
- **Promoção recente (0 anos)** apresentou taxa elevada de desligamento, sugerindo que promoção isoladamente pode não ser suficiente como ação de retenção (aqui trata-se de uma observação descritiva, mas sem inferência causal).

### 5.1 Segmentos prioritários para ações de retenção (Risco x Impacto)

A análise combinada dos principais segmentos (**horas extras**, **renda mensal** e **nível do cargo**) permitiram identificar **perfis** com risco desproporcionalmente alto de desligamento, apoiando a priorização de esforços preventivos de retenção.

### 5.2 Segmentos com maior risco de desligamento (Top 3)

- **Nível 1 senioridade + baixa renda + horas extras realizadas** → **60,78%**
- **Nível 1 senioridade + renda média-baixa + horas extras realizadas** → **36,54%**
- **Nível 3 senioridade + renda alta + horas extras realizadas** → **22,92%**

Observa-se que **horas extras** aparece de forma recorrente entre os segmentos mais críticos, sugerindo que **carga de trabalho** pode ser um fator relevante para priorização de casos e direcionamento de ações gerenciais (ex.: acompanhamento individual, replanejamento de capacidade e medidas de prevenção de burnout).

**Interpretação:** novamente ressalto que os resultados desta etapa são **descritivos** e não indicam causalidade. Ainda assim, fornecem evidências de um estudo analítico, que são suficientes para apoiar ações preventivas orientadas por risco, além de estabelecer hipóteses e priorizações a serem utilizadas na etapa de modelagem preditiva.

In [66]:
# Salvando base de dados processada

import os

caminho_saida = "../data/processed/hr_attrition_processado.csv"
os.makedirs(os.path.dirname(caminho_saida), exist_ok=True) # cria pasta, caso não exista
dados.to_csv(caminho_saida, index=False)

print(f"Base processada salva em: {caminho_saida}")
print(f"Dimensões finais: {dados.shape}")

Base processada salva em: ../data/processed/hr_attrition_processado.csv
Dimensões finais: (1470, 34)


---